# 09. Diagnosing the unmonitored slope error: where does the household reactive power go?

On each interval the unmonitored estimate is `Q_est = Q_inv + e`, where `e = Q_base − Q_load` is the household's reactive load relative to its night-time median. 

On the reference we observe `Q_inv` directly, so **`e = Q_est − Q_ref` is observable here** (synthetic data only).

**Exact decomposition** of the slope error for one site-month (`S` = intervals in BOTH fit windows):

```
slope_err = β_b(W_b) − β_t(W_t)
          = sel_b − sel_t            # interval selection: each fit's own window vs the shared intervals
          + β(e on V_b | S)          # household reactive power (cov(V, e) / var(V))
          + volt_term                # meter V vs PV-circuit V (should be ~0)
```

Checks:
1. Does `β_e = cov(V, e)/var(V)` account for the slope error (vs selection or voltage terms)?
2. Is `e` driven by a few influential intervals/spikes (→ robust regression helps) or by a systematic V-correlation
   (→ needs per-day intercepts / covariates)? Compared via a Huber slope of `e` and the influence of the top intervals.
3. Is `e` step-like (appliance switching) or smooth, and does it correlate with time of day, the household's
   active-load deviation, net import, or month?

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bms_sa_review.ami_analysis.lib import ami_fleet_vvar as Fleet
from bms_sa_review.ami_analysis.lib import ami_conformance_plot as ConformancePlot

## 1. Settings

In [ ]:
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]

# Method settings: identical to notebook 07
NIGHT_HOURS = (0, 4)
P_MIN_FRAC = Fleet.P_MIN_FRAC
V_LO, V_HI = 242, 253
MIN_N_WINDOW, MIN_V_SPREAD = 20, 5.0
SLOPE_TOLERANCE_PCT = 20

# Diagnostic settings
N_CONTROL = 1500        # random sample of correctly classified site-months per class (TP, TN) as controls
SEED = 42
STEP_PCT = 5.0          # |Δe| between consecutive 5-min intervals above this (% of rating) counts as a step
TOP_INFL = 0.05         # share of shared-window intervals with the largest influence on β_e
HUBER_K = 1.345         # Huber tuning constant (95% efficiency under normal errors)
SPIKE_RATIO = 0.5       # "spike-driven" if the Huber slope of e keeps < this share of the OLS slope of e

DUCKDB_MEMORY, DUCKDB_THREADS = "8GB", 4

OUT_DIR = Fleet.store_dir() / "vvar_fleet" / f"2024_2025_band_{V_LO}_{V_HI}"
DIAG_DIR = OUT_DIR / "diagnostics_09"
(DIAG_DIR / "points").mkdir(parents=True, exist_ok=True)
KEYS = ["site_id", "year", "month"]

lo, hi = Fleet.REQUIRED_SLOPE * (1 + SLOPE_TOLERANCE_PCT / 100), Fleet.REQUIRED_SLOPE * (1 - SLOPE_TOLERANCE_PCT / 100)
BAND_LO, BAND_HI = min(lo, hi), max(lo, hi)
print(f"Band: [{BAND_LO:.3f}, {BAND_HI:.3f}] %/V | outputs -> {DIAG_DIR}")

## 2. Verdicts and the site-months to diagnose

Positive class = pass (conformant). 

- **FN** = unmonitored fails, reference passes. 
- **FP** = unmonitored passes, reference fails. 

All FN and FP site-months are diagnosed, plus a random sample of TP and TN as controls.
`side_b` says on which side of the band the unmonitored slope fell (too shallow / too steep).

In [ ]:
site_meta = pd.read_parquet(Fleet.Config.store_path("ami_site_metadata_residential"))
eligibility = Fleet.build_eligibility(site_meta)
sites = eligibility.loc[eligibility.eligibility == "eligible", ["site_id", "rating_kw"]]

blind = Fleet.add_fit_status(Fleet.load_results(OUT_DIR / "blind"), min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD)
truth = Fleet.add_fit_status(Fleet.load_results(OUT_DIR / "truth"), min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD)
constant_q = (truth.fit_status == "ok") & (truth.slope == 0) & (truth.slope_stderr == 0)
truth.loc[constant_q, "fit_status"] = "constant_q"

cols = KEYS + ["slope", "n_window", "passed"]
b_pf = ConformancePlot.add_slope_pass_fail(blind.loc[blind.fit_status == "ok"], SLOPE_TOLERANCE_PCT)
t_pf = ConformancePlot.add_slope_pass_fail(truth.loc[truth.fit_status == "ok"], SLOPE_TOLERANCE_PCT)
both = b_pf[cols].merge(t_pf[cols], on=KEYS, suffixes=("_b", "_t"))
both["outcome"] = np.select(
    [both.passed_b & both.passed_t, both.passed_b & ~both.passed_t, ~both.passed_b & both.passed_t],
    ["TP", "FP", "FN"], default="TN")
both["slope_err"] = both.slope_b - both.slope_t

def band_side(s):
    return np.where(s > BAND_HI, "too_shallow", np.where(s < BAND_LO, "too_steep", "in_band"))
both["side_b"], both["side_t"] = band_side(both.slope_b), band_side(both.slope_t)

# Should reproduce 07: TP 14,137 | FP 1,288 | FN 2,823 | TN 56,557 (n = 74,805)
display(both.outcome.value_counts().reindex(["TP", "FP", "FN", "TN"]).rename("site_months").to_frame())
print("Unmonitored slope side, by outcome:")
display(pd.crosstab(both.outcome, both.side_b, margins=True))
print("Reference slope side, by outcome:")
display(pd.crosstab(both.outcome, both.side_t, margins=True))

In [ ]:
errors = both[both.outcome.isin(["FN", "FP"])]
controls = [g.sample(min(N_CONTROL, len(g)), random_state=SEED)
            for _, g in both[both.outcome.isin(["TP", "TN"])].groupby("outcome")]
pick = pd.concat([errors, *controls], ignore_index=True).merge(sites, on="site_id", how="left")
assert pick.rating_kw.notna().all()
print(f"{len(pick):,} site-months selected:")
display(pick.outcome.value_counts().to_frame("site_months"))

## 3. Interval data for the selected site-months

In [ ]:
def fetch_points(month: str, site_df: pd.DataFrame, source: str) -> pd.DataFrame:
    # Interval-level P/Q estimate (% of rating), meter baseline (% of rating), and the fit-window flag
    con = Fleet.connect(DUCKDB_MEMORY, DUCKDB_THREADS)
    try:
        con.register("_sites", site_df[["site_id", "rating_kw"]])
        return con.execute(f'''
            WITH {Fleet._estimate_sql(month, source, NIGHT_HOURS)}
            SELECT e.site_id, e.t_stamp, e.V,
                   100.0 * e.p_est / s.rating_kw      AS p_pct,
                   100.0 * e.q_est / s.rating_kw      AS q_pct,
                   100.0 * b.base_p_kw / s.rating_kw  AS base_p_pct,
                   coalesce({Fleet._window_sql(source, P_MIN_FRAC, V_LO, V_HI)}, false) AS in_window
            FROM est e
            JOIN _sites s USING (site_id)
            LEFT JOIN baseline b USING (circuit_id)
            WHERE {Fleet._window_sql(source, P_MIN_FRAC, V_LO, V_HI)}   -- only the intervals this fit uses
        ''').df()
    finally:
        con.close()


def month_points(month: str, pick_month: pd.DataFrame) -> pd.DataFrame:
    # Unmonitored and reference intervals side by side for one month (cached)
    path = DIAG_DIR / "points" / f"{month}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    s = pick_month[["site_id", "rating_kw"]].drop_duplicates("site_id")
    b = fetch_points(month, s, "meter_baseline")
    t = fetch_points(month, s, "ground_truth").drop(columns="base_p_pct")
    for name, d in [("unmonitored", b), ("reference", t)]:
        n_dup = d.duplicated(["site_id", "t_stamp"]).sum()
        assert n_dup == 0, f"{month} {name}: {n_dup} duplicated site-timestamps"
    m = b.merge(t, on=["site_id", "t_stamp"], how="outer", suffixes=("_b", "_t"))
    m.to_parquet(path, index=False)
    return m


parts = []
for month in MONTHS:
    y, mo = int(month[:4]), int(month[5:])
    pm = pick[(pick.year == y) & (pick.month == mo)]
    if pm.empty:
        continue
    t0 = time.time()
    m = month_points(month, pm)
    m["year"], m["month"] = y, mo
    parts.append(m.merge(pm[KEYS + ["outcome"]], on=KEYS))   # keep the selected site-months only
    print(f"{month}: {len(pm):4d} site-months, {len(m):9,d} intervals ({time.time() - t0:5.1f} s)")

points = pd.concat(parts, ignore_index=True)
del parts

In [ ]:
# Derived interval quantities
for c in ["in_window_b", "in_window_t"]:
    points[c] = points[c].astype("boolean").fillna(False).astype(bool)

t = points.t_stamp
if getattr(t.dt, "tz", None) is not None:
    t = t.dt.tz_convert("UTC").dt.tz_localize(None)
points["t_utc"] = t
points["hour_aest"] = (t.dt.hour + 10) % 24 + t.dt.minute / 60     # same +10 h shift as the night baseline

# e = Q_est - Q_ref = Q_base - Q_load (household reactive load vs its night median), % of rating
points["e"] = points.q_pct_b - points.q_pct_t

# Reference P in generator convention (the reference gate uses |P|, so fix the sign per site-month)
sign = (points[points.in_window_t].groupby(KEYS).p_pct_t.median().pipe(np.sign).rename("p_sign").reset_index())
points = points.merge(sign, on=KEYS, how="left")
points["p_ref_pct"] = points.p_pct_t * points.p_sign

# e_p = P_est - P_ref = P_base - P_load (household active load vs its night median); net import = P_base - P_est
points["e_p"] = points.p_pct_b - points.p_ref_pct
points["p_net_pct"] = points.base_p_pct - points.p_pct_b

# Sanity: the recomputed window sizes must match the stored fits
n_chk = (points.groupby(KEYS)[["in_window_b", "in_window_t"]].sum().reset_index()
         .merge(pick[KEYS + ["n_window_b", "n_window_t"]], on=KEYS))
print(f"window size matches stored fit: unmonitored {(n_chk.in_window_b == n_chk.n_window_b).mean():.1%}, "
      f"reference {(n_chk.in_window_t == n_chk.n_window_t).mean():.1%} of {len(n_chk):,} site-months")
print(f"{len(points):,} intervals; {(points.in_window_b & points.in_window_t).sum():,} in both windows")

## 4. Per site-month decomposition and residual statistics

For each site-month, on the shared intervals `S`:

| Column | Meaning |
|---|---|
| `sel_b`, `sel_t` | slope over the fit's own window minus slope over `S` (interval-selection effect) |
| `beta_e` | OLS slope of `e` on `V_b` over `S` = `cov(V, e)/var(V)` |
| `volt_term` | slope of `Q_ref` on meter V minus on PV-circuit V (should be ~0) |
| `beta_e_huber` | Huber M-estimate of the slope of `e` (down-weights large residuals) |
| `top_infl_share` | share of `beta_e` contributed by the top `TOP_INFL` most influential intervals |
| `step_frac` | share of consecutive 5-min pairs with `|Δe| > STEP_PCT` |
| `diff_var_ratio` | `var(Δe) / (2 var(e))`: ≈1 white noise, ≈0 smooth/persistent |
| `lag1_autocorr` | lag-1 autocorrelation of `e` |
| `corr_e_ep` | correlation of `e` with the household active-load deviation `e_p` (load switching moves both) |
| `corr_e_V`, `corr_e_hour`, `corr_e_pnet`, `corr_V_hour` | correlations on `S` |

In [ ]:
METRIC_COLS = ["n_b", "n_t", "n_shared", "slope_b_full", "slope_t_full",
               "slope_b_shared", "slope_t_shared", "beta_e", "volt_term", "sel_b", "sel_t", "v_spread_shared",
               "e_mean", "e_sd", "corr_e_V", "corr_e_ep", "corr_e_pnet", "corr_e_hour", "corr_V_hour",
               "beta_e_huber", "top_infl_share", "n_pairs", "step_frac", "diff_var_ratio", "lag1_autocorr"]


def ols_slope(x, y) -> float:
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) < 3:
        return np.nan
    xc = x - x.mean()
    sxx = (xc ** 2).sum()
    return np.nan if sxx == 0 else float((xc * (y - y.mean())).sum() / sxx)


def huber_slope(x, y, k: float = HUBER_K, iters: int = 50) -> float:
    # Huber M-estimate of the slope via IRLS, scale = MAD of residuals (re-estimated each step)
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) < 3 or np.ptp(x) == 0:
        return np.nan
    X = np.column_stack([np.ones_like(x), x - x.mean()])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    for _ in range(iters):
        r = y - X @ beta
        s = np.median(np.abs(r - np.median(r))) / 0.6745
        if s == 0:
            break
        u = np.abs(r) / (k * s)
        w = np.where(u <= 1, 1.0, 1.0 / u)
        WX = X * w[:, None]
        try:
            new = np.linalg.solve(X.T @ WX, WX.T @ y)
        except np.linalg.LinAlgError:
            return np.nan
        if np.allclose(new, beta, rtol=0, atol=1e-9):
            beta = new
            break
        beta = new
    return float(beta[1])


def site_month_metrics(g: pd.DataFrame) -> dict:
    out = dict.fromkeys(METRIC_COLS, np.nan)        # every key always present
    wb, wt = g.in_window_b.values, g.in_window_t.values
    sh = wb & wt
    out.update(n_b=int(wb.sum()), n_t=int(wt.sum()), n_shared=int(sh.sum()),
               slope_b_full=ols_slope(g.V_b[wb], g.q_pct_b[wb]),
               slope_t_full=ols_slope(g.V_t[wt], g.q_pct_t[wt]))
    if sh.sum() < MIN_N_WINDOW:
        return out
    s = g[sh]
    out["slope_b_shared"] = ols_slope(s.V_b, s.q_pct_b)
    out["slope_t_shared"] = ols_slope(s.V_t, s.q_pct_t)
    out["beta_e"] = ols_slope(s.V_b, s.e)
    out["volt_term"] = ols_slope(s.V_b, s.q_pct_t) - out["slope_t_shared"]
    out["sel_b"] = out["slope_b_full"] - out["slope_b_shared"]
    out["sel_t"] = out["slope_t_full"] - out["slope_t_shared"]
    out["v_spread_shared"] = float(np.ptp(s.V_b))

    out["e_mean"], out["e_sd"] = float(s.e.mean()), float(s.e.std())
    out["corr_e_V"] = s.e.corr(s.V_b)
    out["corr_e_ep"] = s.e.corr(s.e_p)
    out["corr_e_pnet"] = s.e.corr(s.p_net_pct)
    out["corr_e_hour"] = s.e.corr(s.hour_aest)
    out["corr_V_hour"] = s.V_b.corr(s.hour_aest)

    out["beta_e_huber"] = huber_slope(s.V_b, s.e)
    xc = (s.V_b - s.V_b.mean()).values
    infl = xc * (s.e - s.e.mean()).values / (xc ** 2).sum()        # sums exactly to beta_e
    top = np.argsort(-np.abs(infl))[: max(1, int(np.ceil(TOP_INFL * len(s))))]
    out["top_infl_share"] = float(infl[top].sum() / infl.sum()) if infl.sum() != 0 else np.nan

    s2 = s.sort_values("t_utc")
    consecutive = (s2.t_utc.diff().dt.total_seconds() == 300).values
    de = s2.e.diff().values[consecutive]
    prev = s2.e.shift().values[consecutive]
    cur = s2.e.values[consecutive]
    out["n_pairs"] = int(consecutive.sum())
    if consecutive.sum() >= 10:
        out["step_frac"] = float((np.abs(de) > STEP_PCT).mean())
        out["diff_var_ratio"] = float(np.var(de) / (2 * np.var(s.e))) if np.var(s.e) > 0 else np.nan
        out["lag1_autocorr"] = (float(np.corrcoef(prev, cur)[0, 1])
                                if np.std(prev) > 0 and np.std(cur) > 0 else np.nan)
    return out


t0 = time.time()
records = []
for key, g in points.groupby(KEYS, sort=False):
    records.append({**dict(zip(KEYS, key)), **site_month_metrics(g)})
metrics = (pd.DataFrame.from_records(records)
           .merge(pick[KEYS + ["outcome", "slope_b", "slope_t", "slope_err", "side_b", "side_t", "rating_kw"]],
                  on=KEYS))
metrics["sel_term"] = metrics.sel_b - metrics.sel_t
metrics["recon_err"] = metrics.sel_term + metrics.beta_e + metrics.volt_term
print(f"{len(metrics):,} site-months in {time.time() - t0:.0f} s; "
      f"{metrics.beta_e.notna().mean():.1%} have >= {MIN_N_WINDOW} shared intervals")

# Checks: recomputed full-window slopes equal the stored fits, and the decomposition is exact
chk = metrics.dropna(subset=["beta_e"])
print(f"max |recomputed - stored| slope: unmonitored {np.nanmax(np.abs(chk.slope_b_full - chk.slope_b)):.2e}, "
      f"reference {np.nanmax(np.abs(chk.slope_t_full - chk.slope_t)):.2e} %/V")
print(f"max |decomposition - slope_err|: {np.nanmax(np.abs(chk.recon_err - chk.slope_err)):.2e} %/V")

metrics.to_parquet(DIAG_DIR / "site_month_metrics.parquet", index=False)

## 5. Results

### 5.1 Which term carries the slope error?

If household reactive power is the cause, `beta_e` should carry most of `slope_err` and the selection and voltage terms should be small. 

The table shows medians by outcome. FN/FP are also split by the side of the band the slope fell on.

In [ ]:
d = metrics.dropna(subset=["beta_e"]).copy()
d["group"] = np.where(d.outcome == "FN", "FN, unmon. " + d.side_b,
             np.where(d.outcome == "FP", "FP, ref. " + d.side_t, d.outcome))
terms = ["slope_err", "beta_e", "sel_term", "volt_term"]

summary = d.groupby("group").agg(n=("slope_err", "size"),
                                 **{f"median_{c}": (c, "median") for c in terms},
                                 **{f"median_abs_{c}": (c, lambda x: x.abs().median()) for c in terms})
display(summary.round(3))

# Share of |slope_err| explained by each term (sum of |term| over sum of |slope_err|), errors only
err = d[d.outcome.isin(["FN", "FP"])]
print("Share of total |slope error| (FN + FP):",
      {c: f"{err[c].abs().sum() / err.slope_err.abs().sum():.1%}" for c in terms[1:]})
r2 = 1 - ((err.slope_err - err.beta_e) ** 2).sum() / ((err.slope_err - err.slope_err.mean()) ** 2).sum()
print(f"R² of slope_err explained by beta_e alone (FN + FP): {r2:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), dpi=130)
colors = {"TP": "#888780", "TN": "#B4B2A9", "FP": "#1f4e79", "FN": "#D85A30"}
for o in ["TP", "TN", "FP", "FN"]:
    g = d[d.outcome == o]
    axes[0].scatter(g.beta_e, g.slope_err, s=5, alpha=0.4, color=colors[o], label=f"{o} (n={len(g):,})")
lim = np.nanpercentile(np.abs(d[["beta_e", "slope_err"]].values), 99.5)
axes[0].plot([-lim, lim], [-lim, lim], color="k", lw=0.8, ls="--")
axes[0].set(xlim=(-lim, lim), ylim=(-lim, lim), xlabel="β_e = cov(V, e) / var(V)  (%/V)",
            ylabel="Slope error, unmonitored − reference (%/V)", title="Household reactive power term vs slope error")
axes[0].legend(fontsize=8, markerscale=3)

for o in ["FN", "FP"]:
    g = d[d.outcome == o]
    axes[1].scatter(g.sel_term, g.slope_err - g.beta_e, s=5, alpha=0.4, color=colors[o], label=o)
axes[1].axhline(0, color="k", lw=0.6); axes[1].axvline(0, color="k", lw=0.6)
axes[1].set(xlabel="Selection term (%/V)", ylabel="slope_err − β_e (%/V)",
            title="What β_e leaves unexplained")
axes[1].legend(fontsize=8, markerscale=3)
fig.tight_layout()
fig.savefig(DIAG_DIR / "decomposition.png", bbox_inches="tight")
plt.show()

### 5.2 Spikes and leverage, or systematic V-correlation?

- **Spike-driven (mechanism b):** a few intervals carry most of `beta_e` (`top_infl_share` high) and the Huber slope of
  `e` is much smaller than the OLS slope. Robust regression should fix these.
- **Systematic (mechanism c):** `beta_e` survives the Huber fit. Robust regression will not help; per-day intercepts,
  covariates or first differences are needed.

Heuristic label for each FN/FP site-month: first the dominant term (`household_q`, `selection`, `voltage`), then, for
`household_q`, `spike` if `beta_e_huber / beta_e < SPIKE_RATIO`, else `systematic`. The thresholds are settings, not results.

In [ ]:
def mechanism(r) -> str:
    mags = {"household_q": abs(r.beta_e), "selection": abs(r.sel_term), "voltage": abs(r.volt_term)}
    top = max(mags, key=mags.get)
    if top != "household_q":
        return top
    ratio = r.beta_e_huber / r.beta_e if r.beta_e != 0 else np.nan
    return "household_q: spike" if ratio < SPIKE_RATIO else "household_q: systematic"

d["huber_ratio"] = d.beta_e_huber / d.beta_e
err = d[d.outcome.isin(["FN", "FP"])].copy()
err["mechanism"] = err.apply(mechanism, axis=1)
display(pd.crosstab(err.group, err.mechanism, margins=True))
display(pd.crosstab(err.group, err.mechanism, normalize="index").round(3))

display(d.groupby("group")[["top_infl_share", "huber_ratio"]].median().round(3)
        .rename(columns=lambda c: f"median_{c}"))

# Would a robust slope of e alone have flipped the verdict? (counterfactual: reference + Huber part of e)
err["slope_b_if_huber"] = err.slope_b - err.beta_e + err.beta_e_huber
err["passes_if_huber"] = err.slope_b_if_huber.between(BAND_LO, BAND_HI)
print("Share of errors whose verdict would match the reference if e's slope were the Huber slope "
      "(indicative only; ignores selection):")
err["fixed_if_huber"] = err.passes_if_huber == (err.outcome == "FN")
display(err.groupby("outcome").fixed_if_huber.mean().round(3).rename("share_fixed").to_frame())
err[KEYS + ["outcome", "group", "mechanism", "slope_b", "slope_t", "beta_e", "beta_e_huber", "sel_term"]].to_csv(
    DIAG_DIR / "error_mechanisms.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), dpi=130)
for o in ["TP", "TN", "FP", "FN"]:
    g = d[d.outcome == o]
    axes[0].hist(g.top_infl_share.clip(-1, 2), bins=60, histtype="step", density=True, color=colors[o], label=o)
    axes[1].hist(g.huber_ratio.clip(-1, 2), bins=60, histtype="step", density=True, color=colors[o], label=o)
    axes[2].hist(g.step_frac.dropna(), bins=40, histtype="step", density=True, color=colors[o], label=o)
axes[0].set(xlabel=f"Share of β_e from top {TOP_INFL:.0%} intervals (clipped)", ylabel="Density")
axes[1].set(xlabel="Huber slope of e / OLS slope of e (clipped)")
axes[2].set(xlabel=f"Share of 5-min pairs with |Δe| > {STEP_PCT:g}% of rating")
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(DIAG_DIR / "spike_vs_systematic.png", bbox_inches="tight")
plt.show()

### 5.3 What does `e` look like and what does it track?

- **Step-like vs smooth:** `step_frac`, `diff_var_ratio` and `lag1_autocorr` by group.
- **Load switching:** `corr_e_ep` > 0 means reactive and active household load move together (e.g. motors).
- **Time of day and voltage:** median `e` and median `V` by AEST hour on the shared intervals. If `e` is most negative
  at the hours when V is lowest, the slope is pushed shallower (and vice versa).
- **e vs V:** median `e` in 1 V bins. A rising profile = positive `beta_e` = shallower unmonitored slope.

In [ ]:
shape_cols = ["e_sd", "step_frac", "diff_var_ratio", "lag1_autocorr",
              "corr_e_ep", "corr_e_V", "corr_e_hour", "corr_e_pnet", "corr_V_hour", "v_spread_shared", "n_shared"]
display(d.groupby("group")[shape_cols].median().round(3))

In [ ]:
shared = points[points.in_window_b & points.in_window_t].merge(d[KEYS + ["group"]], on=KEYS)
shared["hour_bin"] = np.floor(shared.hour_aest * 2) / 2
shared["v_bin"] = np.floor(shared.V_b) + 0.5
show = ["FN, unmon. too_shallow", "FN, unmon. too_steep", "FP, ref. too_shallow", "TP", "TN"]
style = {"FN, unmon. too_shallow": ("#D85A30", "-"), "FN, unmon. too_steep": ("#D85A30", "--"),
         "FP, ref. too_shallow": ("#1f4e79", "-"), "TP": ("#888780", "-"), "TN": ("#B4B2A9", "--")}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), dpi=130)
for grp in show:
    g = shared[shared.group == grp]
    if g.empty:
        continue
    c, ls = style[grp]
    h = g.groupby("hour_bin").agg(e=("e", "median"), V=("V_b", "median"), n=("e", "size"))
    h = h[h.n >= 50]
    axes[0].plot(h.index, h.e, color=c, ls=ls, label=grp)
    axes[1].plot(h.index, h.V, color=c, ls=ls, label=grp)
    v = g.groupby("v_bin").agg(e=("e", "median"), n=("e", "size"))
    v = v[v.n >= 50]
    axes[2].plot(v.index, v.e - v.e.mean(), color=c, ls=ls, marker="o", ms=3, label=grp)
axes[0].set(xlabel="Hour (AEST)", ylabel="Median e (% of rating)", title="Household reactive power vs time of day")
axes[1].set(xlabel="Hour (AEST)", ylabel="Median V (V)", title="Voltage vs time of day (shared intervals)")
axes[2].set(xlabel="Voltage bin (V)", ylabel="Median e, demeaned (% of rating)", title="e vs V (rising = shallower)")
for ax in (axes[0], axes[2]):
    ax.axhline(0, color="k", lw=0.5)
for ax in axes:
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(DIAG_DIR / "e_profiles.png", bbox_inches="tight")
plt.show()

In [ ]:
# e vs the household active-load deviation e_p (FN only, pooled intervals): do Q and P load move together?
g = shared[shared.group.str.startswith("FN")]
fig, ax = plt.subplots(figsize=(5.5, 4.5), dpi=130)
hb = ax.hexbin(g.e_p, g.e, gridsize=60, bins="log", extent=(-100, 60, -40, 40), cmap="Blues")
ax.set(xlabel="e_p = P_base − P_load (% of rating)", ylabel="e = Q_base − Q_load (% of rating)",
       title=f"FN intervals (n = {len(g):,})")
ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
fig.colorbar(hb, ax=ax, label="log10 count")
fig.tight_layout()
plt.show()
print(f"Pooled corr(e, e_p) on FN intervals: {g.e.corr(g.e_p):.3f}; on TP intervals: "
      f"{shared[shared.group == 'TP'].e.corr(shared[shared.group == 'TP'].e_p):.3f}")

## 6. Stability by year and month

Fixes should be chosen on 2024 and reported on 2025, so check that the diagnosis looks the same in both years.

In [ ]:
err["year"] = err.year.astype(int)
display(pd.crosstab([err.year, err.outcome], err.mechanism, normalize="index").round(3))
display(d.groupby(["year", "group"])[["beta_e", "sel_term", "top_infl_share", "huber_ratio", "corr_e_ep"]]
        .median().round(3))

monthly = d.groupby(["year", "month", "outcome"]).beta_e.median().unstack("outcome")
ax = monthly.plot(figsize=(11, 3.5), marker="o", ms=3, color=[colors[c] for c in monthly.columns])
ax.axhline(0, color="k", lw=0.5)
ax.set(xlabel="(year, month)", ylabel="Median β_e (%/V)", title="Household reactive power slope term by month")
plt.tight_layout()
plt.savefig(DIAG_DIR / "beta_e_by_month.png", bbox_inches="tight")
plt.show()

## 7. Examples

Two FN site-months of each kind (largest `|beta_e|`): left, Q vs V for both sources with their fits; right, `e` over
time on the shared intervals for the day with the most of them.

## 7.1. Extreme cases

In [ ]:
N_EXAMPLES = 2
ex = pd.concat([err[(err.outcome == "FN") & (err.mechanism == m)].assign(a=lambda x: x.beta_e.abs())
                .nlargest(N_EXAMPLES, "a") for m in ["household_q: systematic", "household_q: spike"]])

for _, r in ex.iterrows():
    g = points[(points.site_id == r.site_id) & (points.year == r.year) & (points.month == r.month)]
    s = g[g.in_window_b & g.in_window_t].sort_values("t_utc")
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), dpi=120)
    for col, vcol, c, lab in [("q_pct_b", "V_b", "#D85A30", "unmonitored"), ("q_pct_t", "V_t", "#1f4e79", "reference")]:
        w = g[g[f"in_window_{col[-1]}"]]
        axes[0].scatter(w[vcol], w[col], s=6, alpha=0.4, color=c, label=f"{lab}: {ols_slope(w[vcol], w[col]):.2f} %/V")
        xs = np.array([V_LO, V_HI]); b = ols_slope(w[vcol], w[col]); a = w[col].mean() - b * w[vcol].mean()
        axes[0].plot(xs, a + b * xs, color=c)
    axes[0].set(xlabel="V", ylabel="Q (% of rating)", title=f"{r.site_id} {int(r.year)}-{int(r.month):02d}: {r.mechanism}")
    axes[0].legend(fontsize=8)
    day = s.t_utc.dt.floor("D").value_counts().idxmax()
    sd = s[s.t_utc.dt.floor("D") == day]
    axes[1].plot(sd.hour_aest, sd.e, color="#D85A30", marker=".", label="e (% of rating)")
    ax2 = axes[1].twinx()
    ax2.plot(sd.hour_aest, sd.V_b, color="#888780", lw=1, label="V")
    axes[1].set(xlabel="Hour (AEST)", ylabel="e (% of rating)", title=f"{day.date()} (UTC day)")
    ax2.set_ylabel("V")
    fig.tight_layout()
    plt.show()

## 7.2. Typical cases

In [ ]:
# ---------------------------------------------------------------------------
# Typical (not extreme) site-months: |slope_err| between the 40th and 60th percentile
# of each group, sampled at random (seeded). Third panel: e vs V with the beta_e line.
# Day panel uses the local AEST day and breaks lines at gaps.
# ---------------------------------------------------------------------------
N_TYPICAL = 2                      # site-months per group
TYPICAL_Q = (0.40, 0.60)           # |slope_err| quantile range that counts as "typical"
TYPICAL_GROUPS = ["FN, unmon. too_shallow", "FN, unmon. too_steep", "FP, ref. too_shallow", "TP"]

mech = err[KEYS + ["mechanism"]] if "mechanism" in err else pd.DataFrame(columns=KEYS + ["mechanism"])
pool = d.merge(mech, on=KEYS, how="left")

typical = []
for grp in TYPICAL_GROUPS:
    g = pool[pool.group == grp]
    if g.empty:
        continue
    lo_q, hi_q = g.slope_err.abs().quantile(list(TYPICAL_Q))
    mid = g[g.slope_err.abs().between(lo_q, hi_q)]
    typical.append(mid.sample(min(N_TYPICAL, len(mid)), random_state=SEED))
typical = pd.concat(typical, ignore_index=True)
display(typical[KEYS + ["group", "mechanism", "slope_b", "slope_t", "slope_err", "beta_e", "sel_term",
                        "volt_term", "n_shared"]].round(3))


def with_gaps(df: pd.DataFrame, tcol: str, max_gap_min: float = 5) -> pd.DataFrame:
    # Insert a NaN row wherever consecutive timestamps are more than max_gap_min apart (breaks the line)
    df = df.sort_values(tcol)
    gap = df[tcol].diff() > pd.Timedelta(minutes=max_gap_min)
    breaks = df.loc[gap].assign(**{c: np.nan for c in df.columns if c != tcol})
    breaks[tcol] = breaks[tcol] - pd.Timedelta(seconds=1)
    return pd.concat([df, breaks]).sort_values(tcol)


for _, r in typical.iterrows():
    g = points[(points.site_id == r.site_id) & (points.year == r.year) & (points.month == r.month)].copy()
    g["t_aest"] = g.t_utc + pd.Timedelta(hours=10)
    s = g[g.in_window_b & g.in_window_t]

    fig, axes = plt.subplots(1, 3, figsize=(17, 3.9), dpi=120)

    # (1) Q vs V, both sources, with their own fit windows and OLS lines
    axes[0].axhspan(-60, 0, color="none")
    for col, vcol, c, lab in [("q_pct_b", "V_b", "#D85A30", "unmonitored"), ("q_pct_t", "V_t", "#1f4e79", "reference")]:
        w = g[g[f"in_window_{col[-1]}"]]
        b = ols_slope(w[vcol], w[col]); a = w[col].mean() - b * w[vcol].mean()
        axes[0].scatter(w[vcol], w[col], s=6, alpha=0.35, color=c, lw=0, label=f"{lab}: {b:.2f} %/V")
        xs = np.array([V_LO, V_HI]); axes[0].plot(xs, a + b * xs, color=c)
    mech_txt = f" | {r.mechanism}" if isinstance(r.mechanism, str) else ""
    axes[0].set(xlabel="V", ylabel="Q (% of rating)",
                title=f"{r.site_id} {int(r.year)} \n {int(r.month):02d}: {r.group}{mech_txt}")
    axes[0].legend(fontsize=8, title=f"band [{BAND_LO:.2f}, {BAND_HI:.2f}] %/V", title_fontsize=7)

    # (2) e vs V on the shared intervals, with the beta_e line (= the household-Q part of the slope error)
    be = ols_slope(s.V_b, s.e); ae = s.e.mean() - be * s.V_b.mean()
    axes[1].scatter(s.V_b, s.e, s=6, alpha=0.35, color="#888780", lw=0)
    xs = np.array([V_LO, V_HI]); axes[1].plot(xs, ae + be * xs, color="#D85A30")
    axes[1].axhline(0, color="k", lw=0.5)
    axes[1].set(xlabel="V (meter)", ylabel="e = Q_est − Q_ref (% of rating)",
                title=f"β_e = {be:.2f} %/V   (slope error {r.slope_err:.2f}, selection {r.sel_term:.2f}, "
                      f"voltage {r.volt_term:.2f})")

    # (3) one local (AEST) day: e and V over time, lines broken at gaps
    day = s.t_aest.dt.floor("D").value_counts().idxmax()
    sd = with_gaps(s.loc[s.t_aest.dt.floor("D") == day, ["t_aest", "e", "V_b"]], "t_aest")
    hrs = sd.t_aest.dt.hour + sd.t_aest.dt.minute / 60
    axes[2].plot(hrs, sd.e, color="#D85A30", marker=".", ms=3, lw=1, label="e")
    axes[2].axhline(0, color="k", lw=0.5)
    ax2 = axes[2].twinx()
    ax2.plot(hrs, sd.V_b, color="#888780", lw=1, label="V")
    axes[2].set(xlabel="Hour (AEST)", ylabel="e (% of rating)", title=f"{day.date()} (AEST day)")
    ax2.set_ylabel("V")

    fig.tight_layout()
    plt.show()

## 8. Reading the results

| If … | then the main mechanism is … | candidate fix (test in 06b; choose on 2024, report on 2025) |
|---|---|---|
| `beta_e` ≈ `slope_err`, Huber ratio low, high `top_infl_share`, high `step_frac` | spikes / leverage (b) | robust regression (Huber, Theil–Sen); step screening |
| `beta_e` ≈ `slope_err`, Huber ratio ≈ 1, `e` varies with hour or `e_p` | systematic V-correlated load (c) | per-day intercepts; `P_net` / time-of-day covariates; first differences |
| `sel_term` large | interval selection by the signed gate | compare gates; fit on intervals passing a stricter gate |
| most errors close to the band edge regardless | noise near the threshold | "inconclusive" class from the slope confidence interval |

## 9. Huber vs OLS on the full validation set

The unmonitored slope is refitted with a Huber M-estimator (section 4) on **exactly the same intervals** as the OLS fit
(same night baseline, gate, voltage window) for every assessable site-month, not only the 7,111 diagnosed above.
Coverage cannot change; only the unmonitored verdict can.

- The **reference** (PV circuit, OLS) and **timestamp** verdicts are taken unchanged from notebook 06
  (`three_way_site_month.csv`, constant-Q exclusion applied), so the comparison isolates the fitting method.
- `HUBER_KS` gives a sensitivity on the tuning constant (k = 1.345 is the main value; k → ∞ is OLS).
- Choose on 2024, report on 2025 (section 9.4).
- Huber slopes are cached per month in `diagnostics_09/huber_slopes/` (delete a file to recompute that month).

In [ ]:
from bms_sa_review.ami_analysis.lib import ami_vvar_validation as Valid

POSITIVE_IS_FAIL = False                    # positive class = conformant, as in 06/07
HUBER_KS = [1.0, HUBER_K, 2.0]              # tuning-constant sensitivity (main = HUBER_K)
MAIN_K = f"Huber k={HUBER_K:g}"
VALID_DIR = Fleet.store_dir() / "vvar_validation" / "all_sites" / f"band_{V_LO}_{V_HI}"
HUBER_DIR = DIAG_DIR / "huber_slopes"
HUBER_DIR.mkdir(exist_ok=True)

combined_ols = pd.read_csv(VALID_DIR / "three_way_site_month.csv")
for c in ["pass_blind", "pass_truth", "pass_sa"]:
    combined_ols[c] = combined_ols[c].astype("boolean")

blind_ok = blind.loc[blind.fit_status == "ok", KEYS + ["slope", "n_window"]].merge(sites, on="site_id")
print(f"Unmonitored assessable site-months: {len(blind_ok):,} (07) vs "
      f"{combined_ols.pass_blind.notna().sum():,} (06 three-way file)")

In [ ]:
def fetch_window(month: str, site_df: pd.DataFrame) -> pd.DataFrame:
    # The unmonitored fit's own intervals (x = V, y = Q_est % of rating), exactly as month_fit_sql
    con = Fleet.connect(DUCKDB_MEMORY, DUCKDB_THREADS)
    try:
        con.register("_sites", site_df[["site_id", "rating_kw"]])
        return con.execute(f'''
            WITH {Fleet._estimate_sql(month, "meter_baseline", NIGHT_HOURS)}
            SELECT e.site_id, e.V AS x, 100.0 * e.q_est / s.rating_kw AS y
            FROM est e JOIN _sites s USING (site_id)
            WHERE {Fleet._window_sql("meter_baseline", P_MIN_FRAC, V_LO, V_HI)}
        ''').df()
    finally:
        con.close()


parts = []
for month in MONTHS:
    y, mo = int(month[:4]), int(month[5:])
    path = HUBER_DIR / f"{month}.parquet"
    if path.exists():
        parts.append(pd.read_parquet(path))
        continue
    s = blind_ok[(blind_ok.year == y) & (blind_ok.month == mo)].drop_duplicates("site_id")
    if s.empty:
        continue
    t0 = time.time()
    pts = fetch_window(month, s)
    recs = []
    for sid, g in pts.groupby("site_id", sort=False):
        x, yv = g.x.to_numpy(float), g.y.to_numpy(float)
        rec = {"site_id": sid, "year": y, "month": mo, "n": len(x), "slope_ols": ols_slope(x, yv)}
        for k in HUBER_KS:
            rec[f"slope_huber_{k:g}"] = huber_slope(x, yv, k=k)
        recs.append(rec)
    out = pd.DataFrame(recs)
    out.to_parquet(path, index=False)
    parts.append(out)
    print(f"{month}: {len(out):5,d} site-months, {len(pts):10,d} intervals ({time.time() - t0:5.1f} s)")

huber = pd.concat(parts, ignore_index=True)

# Checks: same site-months, same intervals, and the recomputed OLS slope equals the stored one
chk = blind_ok.merge(huber, on=KEYS, how="left")
print(f"{len(huber):,} site-months refitted; missing: {chk.slope_ols.isna().sum()}; "
      f"n matches n_window: {(chk.n == chk.n_window).mean():.1%}; "
      f"max |OLS recomputed - stored|: {np.nanmax(np.abs(chk.slope_ols - chk.slope)):.2e} %/V")

### 9.1 Agreement: OLS (as is) vs Huber

Same tables and confusion matrices as notebook 06, once per fitting method. `OLS (recomputed)` must reproduce
`OLS (as is)` exactly (a check that the refit uses the same data). The reference-vs-timestamp rows do not depend on
the unmonitored fit, so they are identical for every method.

In [ ]:
def with_blind_slope(slope_col: str) -> pd.DataFrame:
    # The 06 three-way table with the unmonitored slope/verdict replaced by `slope_col`
    c = combined_ols.drop(columns=["slope_blind", "pass_blind"]).merge(
        huber[KEYS + [slope_col]].rename(columns={slope_col: "slope_blind"}), on=KEYS, how="left")
    pf = ConformancePlot.add_slope_pass_fail(c[["slope_blind"]].rename(columns={"slope_blind": "slope"}),
                                             SLOPE_TOLERANCE_PCT)
    c["pass_blind"] = pf.passed.astype("boolean").mask(c.slope_blind.isna())
    return c


methods = {"OLS (as is)": combined_ols, "OLS (recomputed)": with_blind_slope("slope_ols"),
           **{f"Huber k={k:g}": with_blind_slope(f"slope_huber_{k:g}") for k in HUBER_KS}}

positive = "non-conformant" if POSITIVE_IS_FAIL else "conformant"
rows = []
for name, comb in methods.items():
    for common_only in (False, True):
        t = Valid.agreement_table(comb, positive_is_fail=POSITIVE_IS_FAIL, common_only=common_only)
        rows.append(t.assign(method=name, overlap="all three sources" if common_only else "each pair"))
comparison = pd.concat(rows, ignore_index=True)
comparison.to_csv(DIAG_DIR / "huber_vs_ols_agreement.csv", index=False)

cols = ["n", "TP", "FP", "FN", "TN", "pass_rate_tested_%", "pass_rate_reference_%",
        "accuracy", "precision", "recall", "specificity", "f1"]
print(f"Positive class = {positive}")
for overlap in ["each pair", "all three sources"]:
    print(f"\n--- {overlap} ---")
    display(comparison[comparison.overlap == overlap]
            .set_index(["tested", "reference", "method"])[cols].sort_index(level=[0, 1], sort_remaining=False)
            .round(3))

In [ ]:
for name in ["OLS (as is)", MAIN_K]:
    fig = Valid.plot_confusion_matrices(methods[name])
    fig.suptitle(name, y=1.03, fontsize=11)
    plt.show()

### 9.2 Which verdicts changed?

For site-months assessed by the unmonitored method and the reference: how many verdicts Huber changes, and whether
each change **fixes** a disagreement with the reference or **breaks** an agreement. Net = fixed − broken.
The same against the timestamp method (`sa`).

In [ ]:
def transitions(reference: str, k_name: str = MAIN_K) -> pd.Series:
    a = methods["OLS (as is)"].dropna(subset=["pass_blind", f"pass_{reference}"])[KEYS + ["pass_blind", f"pass_{reference}"]]
    b = methods[k_name].dropna(subset=["pass_blind"])[KEYS + ["pass_blind"]]
    d = a.merge(b, on=KEYS, suffixes=("_ols", "_huber"))
    ref = d[f"pass_{reference}"].astype(bool)
    ok_ols = d.pass_blind_ols.astype(bool) == ref
    ok_hub = d.pass_blind_huber.astype(bool) == ref
    return pd.Series({
        "n": len(d),
        "verdict changed": int((d.pass_blind_ols != d.pass_blind_huber).sum()),
        "fixed (wrong -> right)": int((~ok_ols & ok_hub).sum()),
        "broken (right -> wrong)": int((ok_ols & ~ok_hub).sum()),
        "net fixed": int((~ok_ols & ok_hub).sum() - (ok_ols & ~ok_hub).sum()),
        "FN fixed": int((~ok_ols & ok_hub & ref).sum()),
        "FP fixed": int((~ok_ols & ok_hub & ~ref).sum()),
        "new FN": int((ok_ols & ~ok_hub & ref).sum()),
        "new FP": int((ok_ols & ~ok_hub & ~ref).sum()),
    })

display(pd.DataFrame({f"{k} vs {ref}": transitions(ref, k)
                      for ref in ["truth", "sa"] for k in [f"Huber k={x:g}" for x in HUBER_KS]}).T)

### 9.3 Fleet conformance under each method

Huber also changes the fleet-level rates reported in the paper (07). Site level = conformant in every assessable month.

In [ ]:
def fleet_rates(comb: pd.DataFrame) -> pd.Series:
    b = comb.dropna(subset=["pass_blind"])
    site = b.groupby("site_id").pass_blind.apply(lambda s: s.astype(bool).all())
    return pd.Series({"site-months": len(b), "site-month conformant %": 100 * b.pass_blind.astype(bool).mean(),
                      "sites": len(site), "site conformant %": 100 * site.mean()})

display(pd.DataFrame({name: fleet_rates(comb) for name, comb in methods.items()}).T.round(2))

### 9.4 By year (choose on 2024, report on 2025)

Unmonitored vs reference, each pair's own overlap.

In [ ]:
rows = []
for name, comb in methods.items():
    for year, g in comb.groupby("year"):
        t = Valid.agreement_table(g, positive_is_fail=POSITIVE_IS_FAIL, pairs=[("blind", "truth")])
        rows.append(t.assign(method=name, year=int(year)))
by_year = pd.concat(rows, ignore_index=True)
display(by_year.pivot_table(index="method", columns="year", values=["accuracy", "f1", "FP", "FN"])
        .reindex(list(methods)).round(3))

### 9.5 How much does Huber move the slope?

Left: Huber against OLS slope for every assessable site-month (pass band shaded). Points off the diagonal are where a
few intervals were pulling the OLS line. Right: distribution of the change.

In [ ]:
h = huber.dropna(subset=["slope_ols", f"slope_huber_{HUBER_K:g}"])
dh = h[f"slope_huber_{HUBER_K:g}"] - h.slope_ols

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), dpi=130)
lim = (-8, 2)
axes[0].axvspan(BAND_LO, BAND_HI, color="#1baf7a", alpha=0.12, lw=0)
axes[0].axhspan(BAND_LO, BAND_HI, color="#1baf7a", alpha=0.12, lw=0)
axes[0].scatter(h.slope_ols, h[f"slope_huber_{HUBER_K:g}"], s=2, alpha=0.2, color="#2a78d6", lw=0)
axes[0].plot(lim, lim, color="k", lw=0.6)
axes[0].set(xlim=lim, ylim=lim, xlabel="OLS slope (%/V)", ylabel=f"Huber slope, k={HUBER_K:g} (%/V)",
            title=f"Unmonitored slope, n = {len(h):,}")
axes[1].hist(dh.clip(-1, 1), bins=200, color="#888780")
axes[1].set(xlabel="Huber − OLS (%/V, clipped at ±1)", ylabel="Site-months", yscale="log",
            title=f"median |Δ| {dh.abs().median():.3f}, p95 {dh.abs().quantile(.95):.3f} %/V")
fig.tight_layout()
fig.savefig(DIAG_DIR / "huber_vs_ols_slopes.png", bbox_inches="tight")
plt.show()